# Lesson 07 - পরিকল্পনা ডিজাইন প্যাটার্ন

এই নোটবুকটি Microsoft Agent Framework ব্যবহার করে AI এজেন্টদের জন্য **পরিকল্পনা ডিজাইন প্যাটার্ন** প্রদর্শন করে।
আপনি শিখবেন কীভাবে জটিল ভ্রমণ অনুরোধগুলিকে কাঠামোবদ্ধ উপকর্মে বিভক্ত করতে হয়, সেগুলো বিশেষজ্ঞ এজেন্টদের কাছে নিয়োগ করতে হয়,
এবং গঠিত পরিকল্পনাটি সম্পাদন করতে হয় — সবকিছুই Pydantic মডেল দ্বারা চালিত কাঠামোবদ্ধ আউটপুট ব্যবহার করে।


## সেটআপ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## টাস্ক বিভাজন

টাস্ক বিভাজন হল পরিকল্পনার ডিজাইন প্যাটার্নের মূল। একটি একক এজেন্টকে জটিল অনুরোধ সম্পূর্ণ করার বদলে, আমরা সমস্যাটিকে ছোট, সুপরিকল্পিত **উপটাস্ক** এ ভাগ করি। প্রতিটি উপটাস্ক বিশেষজ্ঞ এজেন্টের কাছে নিয়োগ দেওয়া হয় (যেমন, ফ্লাইট, হোটেল, কার্যক্রম) স্পষ্ট অগ্রাধিকার এবং নির্ভরশীলতার ক্রম অনুসারে।

এই পদ্ধতিটি বেশ কয়েকটি সুবিধা দেয়:
- **স্পষ্টতা**: প্রতিটি উপটাস্কের একটি নির্দিষ্ট দায়িত্ব থাকে।
- **সমান্তরালতা**: স্বাধীন উপটাস্কগুলি পাশাপাশি চলতে পারে।
- **বিশ্বাসযোগ্যতা**: ব্যর্থতা শুধুমাত্র নির্দিষ্ট উপটাস্কের মধ্যে সীমাবদ্ধ থাকে।
- **বাজেট ট্র্যাকিং**: খরচ প্রতিটি উপটাস্কের জন্য অনুমান করা হয় এবং একত্রিত করা হয়।


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## স্ট্রাকচার্ড আউটপুট সহ একটি পরিকল্পনা এজেন্ট তৈরি করা

পরিকল্পনা এজেন্ট একটি **ফ্রন্ট ডেস্ক কোঅর্ডিনেটর** হিসাবে কাজ করে। একটি উচ্চ-স্তরের ভ্রমণের অনুরোধ দেওয়া হলে এটি একটি স্ট্রাকচার্ড `TravelPlan` তৈরি করে — অনুরোধটিকে উপকার্যে ভাঙা, অগ্রাধিকার নির্ধারণ করা, এবং নির্ভরশীলতাগুলো চিহ্নিত করা যাতে একটি কনসিয়ার্জ বা সম্পাদনা স্তর কাজটি সম্পন্ন করতে পারে।


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## বিশেষজ্ঞ সরঞ্জাম দিয়ে একটি পরিকল্পনা কার্যকর করা

একবার ফ্রন্ট ডেস্ক এজেন্ট একটি কাঠামোবদ্ধ পরিকল্পনা তৈরি করলে, **কনসিয়ার্জ এজেন্ট** এটি কার্যকর করে। প্রতিটি বিশেষজ্ঞ সরঞ্জাম একটি উপ-কর্তব্যের একটি শ্রেণী (উড়ান, হোটেল, কার্যক্রম) সামলায়। কনসিয়ার্জ পরিকল্পনার উপ-কর্তব্যগুলি নির্ভরশীলতার ক্রমে পুনরাবৃত্তি করে এবং প্রতিটি উপ-কর্তব্যকে উপযুক্ত সরঞ্জামে প্রেরণ করে।


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## সারাংশ

এই পাঠে আপনি AI এজেন্টদের জন্য **পরিকল্পনা ডিজাইন প্যাটার্ন** শিখেছেন:

1. **কাজ বিভাজন** — একটি ফ্রন্ট ডেস্ক পরিকল্পনা এজেন্ট একটি জটিল ভ্রমণ অনুরোধকে কাঠামোবদ্ধ উপকাজে ভাগ করে, প্রতিটির জন্য পিড্যান্টিক মডেল ব্যবহার করে বিশেষজ্ঞ এজেন্টকে অগ্রাধিকার এবং নির্ভরতাসহ বরাদ্দ করে।
2. **কাঠামোবদ্ধ আউটপুট** — `response_format` প্রদান করে এজেন্ট একটি বৈধকৃত `TravelPlan` অবজেক্ট ফেরত দেয়, যা অবাধ আকারের টেক্সটের পরিবর্তে পরবর্তী প্রক্রিয়াকরণকে নির্ভরযোগ্য করে তোলে।
3. **পরিকল্পনা কার্যকর করা** — একটি কনসিয়ার্জ এজেন্ট উপকাজগুলো বিশেষজ্ঞ টুলস (`book_flight`, `reserve_hotel`, `book_activity`) ব্যবহার করে পরিকল্পনা সম্পূর্ণ করে এবং ফলাফল প্রতিবেদন করে।

এই প্যাটার্ন *কী করতে হবে* (পরিকল্পনা) থেকে *কিভাবে করতে হবে* (কার্যকর) আলাদা করে, ফলে এজেন্টগুলো আরও মডিউলার, পরীক্ষাযোগ্য এবং সম্প্রসারণযোগ্য হয়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**দ্রষ্টব্য**:  
এই নথিটি AI অনুবাদ সেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা যথাসম্ভব সঠিকতার জন্য চেষ্টা করি, স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে তা দয়া করে জানবেন। মূল নথিটি তার মাতৃভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচনা করা উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদের সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে সৃষ্ট কোনো ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
